In [317]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [318]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [319]:
torch.manual_seed(44)

In [320]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Available devide: {device}")

Available devide: cuda


In [321]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df = df.dropna() # Drop rows with NaN values
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0,0,1,2,0,0,0,0,...,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,3,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [322]:
df.shape

(58160, 785)

In [323]:
print(len(df['label'].unique()))
df['label'].unique()

10


array([2, 9, 6, 0, 3, 4, 5, 8, 7, 1])

In [324]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [325]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44)

In [326]:
# scaling the features:
X_train = X_train/255
X_test = X_test/255

In [327]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [328]:
X_train_data = CustomDataset(X_train, y_train)
X_test_data = CustomDataset(X_test, y_test)

In [329]:
X_train_dataloader = DataLoader(X_train_data, batch_size=128, shuffle=True, pin_memory=True)
X_test_dataloader = DataLoader(X_test_data, batch_size=128, shuffle=False, pin_memory=True)

In [330]:
class ToqeersNN(nn.Module):
    def __init__(self, num_features):
        super(ToqeersNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )


    def forward(self, X):
        return self.model(X)

In [331]:
epochs = 200
learning_rate = 0.1

In [332]:
model = ToqeersNN(num_features=X_train.shape[1]).to(device)

In [333]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [334]:
for epoch in range(epochs):
    learning_rate = learning_rate * (0.98 ** epoch)
    for batch_x, batch_y in X_train_dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        # forword pass:
        output = model(batch_x)
        # calculate loss:
        loss = loss_fn(output, batch_y)
        # claculating gradients:
        loss.backward()
        # updating weights:
        optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss}, learning_rate :{learning_rate}")

Epoch: 1, Loss: 0.7438989877700806, learning_rate :0.1
Epoch: 2, Loss: 0.35008180141448975, learning_rate :0.098
Epoch: 3, Loss: 0.3823449909687042, learning_rate :0.0941192
Epoch: 4, Loss: 0.4718397855758667, learning_rate :0.0885842380864
Epoch: 5, Loss: 0.37468230724334717, learning_rate :0.08170728068875467
Epoch: 6, Loss: 0.3403732180595398, learning_rate :0.07385691026454037
Epoch: 7, Loss: 0.5081526041030884, learning_rate :0.06542558123199924
Epoch: 8, Loss: 0.488781213760376, learning_rate :0.05679761759500593
Epoch: 9, Loss: 0.3737550675868988, learning_rate :0.048321312820571644
Epoch: 10, Loss: 0.3350055515766144, learning_rate :0.04028778642734252
Epoch: 11, Loss: 0.344300776720047, learning_rate :0.03291805473947476
Epoch: 12, Loss: 0.3065323233604431, learning_rate :0.026358518435595345
Epoch: 13, Loss: 0.30590155720710754, learning_rate :0.020683970229283703
Epoch: 14, Loss: 0.33807212114334106, learning_rate :0.01590643620510861
Epoch: 15, Loss: 0.3504429757595062, lea

In [335]:
model.eval()

ToqeersNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [336]:
total = 0
correct = 0
with torch.no_grad():
    for batch_x, batch_y in X_test_dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        output = model(batch_x) # autometically calls forward pass from NNclass.
        _, predicted = torch.max(output.data, 1)
        total += batch_y.shape[0]
        correct += (predicted == batch_y).sum().item()

print(f"Accuracy: {correct/total}")

Accuracy: 0.8917640990371389
